# 05 · Supplementary Figure S1 — Reference M-ComBat quality control

This notebook converts the audited diagnostics from notebook 04 into a publication-ready supplementary figure. It is self-contained within the InterGATE repository: all visual parameters, labels, colours, output formats and panel functions are defined here and can be edited from a single configuration cell.

**Scientific scope.** The figure documents cohort-associated structure before and after reference M-ComBat, class-signal preservation, and the no-leakage projection contract. These diagnostics are descriptive and are not used for endpoint, split, model, κ, or checkpoint selection.

**Required inputs**

- **output_prepared/expr_prepared.csv** and **output_prepared/metadata_prepared.csv** from notebook 02.
- The four M-ComBat matrices/metadata files and **verificacion_invariancia.csv** from notebook 03.

**Primary reference**

Stein CK et al. (2015). *Removing batch effects from purified plasma cell gene expression microarrays with modified ComBat*. BMC Bioinformatics 16, 63. https://doi.org/10.1186/s12859-015-0478-3


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
import math
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from IPython.display import Image, Markdown, display


def locate_data_prepare(start: Path) -> Path:
    """Locate data_prepare whether Jupyter starts in the notebook or repo root."""
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "qc_fuga.py").is_file() and (base / "combat_ref.py").is_file():
            return base
        candidate = base / "data_prepare"
        if (candidate / "qc_fuga.py").is_file() and (candidate / "combat_ref.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate data_prepare/. Start Jupyter inside the InterGATE repository."
    )


DATA_PREPARE = locate_data_prepare(Path.cwd())
if str(DATA_PREPARE) not in sys.path:
    sys.path.insert(0, str(DATA_PREPARE))

from qc_fuga import (
    cargar_salidas,
    comparar_pre_post,
    cramer_v_batch_label,
    reference_pca_scores,
    tabla_composicion,
)

print(f"DATA_PREPARE: {DATA_PREPARE}")


## 1. Editable configuration

This is the only cell that normally needs editing. Cohort colours follow the current manuscript palette; unknown cohorts receive deterministic fallback colours. Set **force_recompute_qc=False** to reuse an existing QC table, or enable TIFF if required by the journal.


In [ ]:
@dataclass(frozen=True)
class FigureConfig:
    # Data contract
    cohort_col: str = "dataset"
    label_col: str = "label"
    n_pc_qc: int = 20
    n_pc_plot: int = 2
    seed: int = 42
    force_recompute_qc: bool = True

    # Canvas and typography
    figure_size: tuple[float, float] = (18.4, 10.8)
    dpi: int = 600
    font_family: str = "DejaVu Sans"
    title_size: float = 20
    panel_title_size: float = 12.5
    label_size: float = 10.5
    tick_size: float = 9.5
    legend_size: float = 8.7

    # Dense PCA layers
    point_size: float = 13
    point_alpha: float = 0.68
    point_edge_width: float = 0.25

    # Output
    output_subdir: str = "figures"
    output_stem: str = "Supplementary_Figure_S1_MComBat_QC"
    export_pdf: bool = True
    export_svg: bool = True
    export_png: bool = True
    export_tiff: bool = False

    # Manuscript-matched palette
    cohort_colors: dict[str, str] = field(default_factory=lambda: {
        "GSE96058": "#4E79A7",
        "GSE25066": "#76B7B2",
        "GSE162228": "#F28E2B",
        "GSE20711": "#C44E52",
        "TCGA-BRCA": "#9C6ADE",
        "GSE21653": "#59A14F",
        "GSE1456": "#EDC948",
    })
    split_colors: dict[str, str] = field(default_factory=lambda: {
        "train": "#2F6FB0",
        "test": "#E07A1F",
    })
    before_color: str = "#A7ADB5"
    after_color: str = "#147D80"
    grid_color: str = "#D9DEE5"
    text_color: str = "#20242A"

    # Raw metrics; no rescaling is applied.
    technical_metrics: tuple[tuple[str, str, str], ...] = (
        ("batch_silhouette", "Cohort silhouette", "lower"),
        ("batch_balanced_accuracy_cv", "Cohort prediction · balanced accuracy", "lower"),
        ("knn_batch_entropy", "Local cohort entropy", "higher"),
        ("gene_batch_eta2_median", "Gene cohort η² · median", "lower"),
        ("gene_batch_eta2_q90", "Gene cohort η² · 90th percentile", "lower"),
    )


CFG = FigureConfig()
FIGURE_DIR = DATA_PREPARE / "output_mcombat" / CFG.output_subdir
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

config_table = pd.Series(asdict(CFG), name="value").to_frame()
display(config_table)
print(f"Outputs will be written to: {FIGURE_DIR}")


## 2. Publication style and shared helpers


In [ ]:
def apply_publication_style(cfg: FigureConfig) -> None:
    mpl.rcParams.update({
        "font.family": cfg.font_family,
        "font.size": cfg.tick_size,
        "axes.titlesize": cfg.panel_title_size,
        "axes.titleweight": "bold",
        "axes.labelsize": cfg.label_size,
        "axes.labelcolor": cfg.text_color,
        "axes.edgecolor": "#2D3136",
        "axes.linewidth": 0.9,
        "xtick.labelsize": cfg.tick_size,
        "ytick.labelsize": cfg.tick_size,
        "xtick.color": cfg.text_color,
        "ytick.color": cfg.text_color,
        "legend.fontsize": cfg.legend_size,
        "legend.frameon": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def panel_title(ax: mpl.axes.Axes, letter: str, title: str) -> None:
    ax.set_title(
        f"{letter}  {title}",
        loc="left",
        pad=9,
        fontsize=CFG.panel_title_size,
        fontweight="bold",
        color=CFG.text_color,
    )


def clean_axis(ax: mpl.axes.Axes, *, grid_axis: str | None = None) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=CFG.grid_color, linewidth=0.65, alpha=0.75)
        ax.set_axisbelow(True)


def align_meta(meta: pd.DataFrame, samples) -> pd.DataFrame:
    """Return metadata aligned exactly to a sequence of sample identifiers."""
    aligned = meta.copy()
    if "sample" in aligned.columns:
        aligned.index = aligned["sample"].astype(str)
    else:
        aligned.index = aligned.index.astype(str)
    sample_index = pd.Index([str(x) for x in samples])
    missing = sample_index.difference(aligned.index)
    if len(missing):
        raise KeyError(f"Metadata are missing {len(missing)} samples; first: {missing[0]}")
    return aligned.loc[sample_index]


def value_at(frame: pd.DataFrame, split: str, stage: str, metric: str) -> float:
    row = frame.loc[(frame["split"] == split) & (frame["stage"] == stage), metric]
    if len(row) != 1:
        raise ValueError(f"Expected one QC row for split={split}, stage={stage}; found {len(row)}")
    return float(row.iloc[0])


def make_palette(levels: list[str], preferred: dict[str, str]) -> dict[str, str]:
    """Deterministic palette with manuscript colours first and tab20 fallbacks."""
    fallback = [mpl.colors.to_hex(c) for c in mpl.colormaps["tab20"].colors]
    palette: dict[str, str] = {}
    cursor = 0
    for level in levels:
        if level in preferred:
            palette[level] = preferred[level]
        else:
            while fallback[cursor % len(fallback)] in preferred.values():
                cursor += 1
            palette[level] = fallback[cursor % len(fallback)]
            cursor += 1
    return palette


apply_publication_style(CFG)


## 3. Load the frozen matrices and recompute the diagnostics

The PCA/scaler is fitted in development separately for the pre- and post-M-ComBat representations. External samples are transformed with the corresponding development fit and are never refitted. The invariance file written by notebook 03 is checked before plotting.


In [ ]:
d = cargar_salidas(DATA_PREPARE)
OUT = DATA_PREPARE / "output_mcombat"

required_meta = {CFG.cohort_col, CFG.label_col}
for split in ("train", "test"):
    missing = required_meta.difference(d[f"meta_{split}"].columns)
    if missing:
        raise KeyError(f"meta_{split} is missing required columns: {sorted(missing)}")

qc_path = OUT / "qc_batch_pre_post.csv"
if CFG.force_recompute_qc or not qc_path.is_file():
    qc = comparar_pre_post(
        **d,
        cohort_col=CFG.cohort_col,
        label_col=CFG.label_col,
        n_pc=CFG.n_pc_qc,
        seed=CFG.seed,
    )
    qc.to_csv(qc_path, index=False)
else:
    qc = pd.read_csv(qc_path)

expected_pairs = {
    ("train", "pre_mcombat"),
    ("train", "post_mcombat"),
    ("test", "pre_mcombat"),
    ("test", "post_mcombat"),
}
observed_pairs = set(qc[["split", "stage"]].itertuples(index=False, name=None))
if observed_pairs != expected_pairs:
    raise ValueError(f"Unexpected QC rows: {sorted(observed_pairs)}")

qc_platform = None
if "platform" in d["meta_all"].columns:
    platform_path = OUT / "qc_platform_pre_post.csv"
    if CFG.force_recompute_qc or not platform_path.is_file():
        qc_platform = comparar_pre_post(
            **d,
            cohort_col="platform",
            label_col=CFG.label_col,
            n_pc=CFG.n_pc_qc,
            seed=CFG.seed,
        )
        qc_platform.to_csv(platform_path, index=False)
    else:
        qc_platform = pd.read_csv(platform_path)

composition = {
    split: tabla_composicion(
        d[f"meta_{split}"],
        cohort_col=CFG.cohort_col,
        label_col=CFG.label_col,
    )
    for split in ("train", "test")
}
confounding = pd.DataFrame([
    {
        "split": split,
        "cramers_v_cohort_label": cramer_v_batch_label(
            d[f"meta_{split}"],
            cohort_col=CFG.cohort_col,
            label_col=CFG.label_col,
        ),
    }
    for split in ("train", "test")
])

invariance_path = OUT / "verificacion_invariancia.csv"
invariance = pd.read_csv(invariance_path)
if "max_dif" not in invariance.columns:
    raise KeyError(f"{invariance_path.name} does not contain a max_dif column.")
max_invariance_delta = float(pd.to_numeric(invariance["max_dif"], errors="raise").max())
assert max_invariance_delta < 1e-9, (
    f"External invariance contract failed: max_dif={max_invariance_delta:.3e}"
)

display(Markdown("### Cohort QC"))
display(qc)
if qc_platform is not None:
    display(Markdown("### Optional platform QC"))
    display(qc_platform)
display(Markdown("### Cohort × class composition"))
display(confounding)
for split, table in composition.items():
    display(Markdown(f"**{split.capitalize()}**"))
    display(table)
display(Markdown(f"**External invariance:** PASS; maximum difference = {max_invariance_delta:.3e}"))


## 4. Construct development-fitted PCA projections

Only coordinates used by the figure are assembled here. The full 20-component representation remains the basis of the quantitative QC metrics above.


In [ ]:
def build_pca_coordinates(data: dict, cfg: FigureConfig):
    genes = data["expr_train"].index.intersection(data["expr_test"].index)
    if len(genes) == 0:
        raise ValueError("Development and external matrices share no genes.")

    train_post = data["expr_train"].loc[genes]
    test_post = data["expr_test"].loc[genes]
    train_pre = data["expr_pre"].loc[genes, train_post.columns.astype(str)]
    test_pre = data["expr_pre"].loc[genes, test_post.columns.astype(str)]

    frames: list[pd.DataFrame] = []
    variance_rows: list[dict] = []
    for stage, train_expr, test_expr in (
        ("pre_mcombat", train_pre, test_pre),
        ("post_mcombat", train_post, test_post),
    ):
        z_train, z_test, explained = reference_pca_scores(
            train_expr,
            test_expr,
            n_pc=cfg.n_pc_plot,
            seed=cfg.seed,
        )
        variance_rows.append({
            "stage": stage,
            "pc1_variance": float(explained[0]),
            "pc2_variance": float(explained[1]) if len(explained) > 1 else np.nan,
        })

        for split, expr, scores in (
            ("train", train_expr, z_train),
            ("test", test_expr, z_test),
        ):
            meta = align_meta(data[f"meta_{split}"], expr.columns)
            frame = pd.DataFrame({
                "sample": [str(x) for x in expr.columns],
                "split": split,
                "stage": stage,
                "PC1": scores[:, 0],
                "PC2": scores[:, 1] if scores.shape[1] > 1 else np.zeros(len(scores)),
                "cohort": meta[cfg.cohort_col].astype(str).to_numpy(),
                "label": meta[cfg.label_col].astype(str).to_numpy(),
            })
            frames.append(frame)

    coordinates = pd.concat(frames, ignore_index=True)
    variance = pd.DataFrame(variance_rows)
    return coordinates, variance


pca_df, pca_variance = build_pca_coordinates(d, CFG)

observed_cohorts = pca_df["cohort"].drop_duplicates().tolist()
preferred_order = [c for c in CFG.cohort_colors if c in observed_cohorts]
cohort_levels = preferred_order + sorted(set(observed_cohorts).difference(preferred_order))
COHORT_PALETTE = make_palette(cohort_levels, CFG.cohort_colors)

display(pca_variance)
display(pca_df.groupby(["split", "stage", "cohort"], observed=True).size().rename("n").to_frame())


## 5. Fully editable panel functions

The point layers are rasterized inside PDF/SVG output to keep file sizes manageable; all text, axes and metric layers remain vector elements.


In [ ]:
def plot_pca_panel(
    ax: mpl.axes.Axes,
    *,
    split: str,
    stage: str,
    letter: str,
    title: str,
) -> None:
    sub = pca_df.loc[(pca_df["split"] == split) & (pca_df["stage"] == stage)].copy()
    if sub.empty:
        raise ValueError(f"No PCA coordinates for {split}/{stage}")

    for cohort in cohort_levels:
        block = sub.loc[sub["cohort"] == cohort]
        if block.empty:
            continue
        ax.scatter(
            block["PC1"],
            block["PC2"],
            s=CFG.point_size,
            alpha=CFG.point_alpha,
            c=COHORT_PALETTE[cohort],
            edgecolors="white",
            linewidths=CFG.point_edge_width,
            rasterized=True,
            label=cohort,
            zorder=2,
        )

    ax.axhline(0, color=CFG.grid_color, linewidth=0.65, zorder=0)
    ax.axvline(0, color=CFG.grid_color, linewidth=0.65, zorder=0)
    var = pca_variance.loc[pca_variance["stage"] == stage].iloc[0]
    ax.set_xlabel(f"PC1 ({100 * var['pc1_variance']:.1f}% development variance)")
    ax.set_ylabel(f"PC2 ({100 * var['pc2_variance']:.1f}% development variance)")
    panel_title(ax, letter, title)
    ax.text(
        0.985,
        0.985,
        f"n = {len(sub):,}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=CFG.legend_size,
        color="#50555B",
    )
    clean_axis(ax)


def plot_technical_metrics(ax: mpl.axes.Axes) -> None:
    n_metrics = len(CFG.technical_metrics)
    y_base = np.arange(n_metrics)[::-1].astype(float)
    offsets = {"train": 0.15, "test": -0.15}
    all_values: list[float] = []

    for row_index, (metric, label, direction) in enumerate(CFG.technical_metrics):
        y = y_base[row_index]
        if row_index % 2 == 0:
            ax.axhspan(y - 0.42, y + 0.42, color="#F5F7F9", zorder=0)

        for split in ("train", "test"):
            pre = value_at(qc, split, "pre_mcombat", metric)
            post = value_at(qc, split, "post_mcombat", metric)
            if not (np.isfinite(pre) and np.isfinite(post)):
                continue
            all_values.extend([pre, post])
            yy = y + offsets[split]
            colour = CFG.split_colors[split]
            ax.plot([pre, post], [yy, yy], color=colour, linewidth=2.0, alpha=0.88, zorder=2)
            ax.scatter(
                pre, yy, s=52, facecolor="white", edgecolor=colour,
                linewidth=1.8, zorder=3,
            )
            ax.scatter(
                post, yy, s=58, facecolor=colour, edgecolor="white",
                linewidth=0.7, zorder=4,
            )
            text_y = 7 if split == "train" else -11
            ax.annotate(
                f"{pre:.2f}",
                (pre, yy),
                xytext=(0, text_y),
                textcoords="offset points",
                ha="center",
                va="center",
                fontsize=7.1,
                color="#666B70",
            )
            ax.annotate(
                f"{post:.2f}",
                (post, yy),
                xytext=(0, text_y),
                textcoords="offset points",
                ha="center",
                va="center",
                fontsize=7.1,
                fontweight="bold",
                color=colour,
            )

    if not all_values:
        raise ValueError("No finite technical metrics are available.")
    xmin = min(-0.08, min(all_values) - 0.08)
    xmax = max(1.02, max(all_values) + 0.08)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(-0.6, n_metrics - 0.4)
    ax.set_yticks(y_base)
    ax.set_yticklabels([f"{x[1]} ({x[2]})" for x in CFG.technical_metrics])
    ax.set_xlabel("Raw diagnostic value · open = before, filled = after")
    ax.axvline(0, color="#8B9096", linestyle=(0, (3, 3)), linewidth=0.8, zorder=1)
    panel_title(ax, "E", "Residual cohort structure")
    clean_axis(ax, grid_axis="x")

    legend = [
        Line2D([0], [0], color=CFG.split_colors["train"], lw=2.2, label="Development"),
        Line2D([0], [0], color=CFG.split_colors["test"], lw=2.2, label="External"),
        Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="white",
               markeredgecolor="#62676D", markeredgewidth=1.4, label="Before"),
        Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=CFG.after_color,
               markeredgecolor="white", label="After"),
    ]
    ax.legend(handles=legend, loc="lower right", bbox_to_anchor=(1.0, 1.01), ncol=4)


def plot_signal_panel(ax: mpl.axes.Axes) -> None:
    rows = [("train", "Development"), ("test", "External")]
    values = []
    for split, label in rows:
        pre = value_at(qc, split, "pre_mcombat", "label_silhouette")
        post = value_at(qc, split, "post_mcombat", "label_silhouette")
        values.extend([pre, post])
        y = 1 if split == "train" else 0
        colour = CFG.split_colors[split]
        ax.plot([pre, post], [y, y], color=colour, linewidth=2.2, zorder=2)
        ax.scatter(pre, y, s=55, facecolor="white", edgecolor=colour, linewidth=1.8, zorder=3)
        ax.scatter(post, y, s=62, facecolor=colour, edgecolor="white", linewidth=0.7, zorder=4)
        ax.annotate(f"{pre:.3f}", (pre, y), xytext=(0, 10), textcoords="offset points",
                    ha="center", fontsize=7.5, color="#666B70")
        ax.annotate(f"{post:.3f}", (post, y), xytext=(0, -14), textcoords="offset points",
                    ha="center", fontsize=7.5, fontweight="bold", color=colour)

    finite = [x for x in values if np.isfinite(x)]
    if finite:
        pad = max(0.03, 0.18 * (max(finite) - min(finite) or 0.1))
        ax.set_xlim(min(finite) - pad, max(finite) + pad)
    ax.set_ylim(-0.55, 1.55)
    ax.set_yticks([1, 0])
    ax.set_yticklabels(["Development", "External"])
    ax.set_xlabel("Class-label silhouette")
    panel_title(ax, "F", "Signal preservation and safeguards")
    clean_axis(ax, grid_axis="x")


def add_card(
    ax: mpl.axes.Axes,
    xy: tuple[float, float],
    width: float,
    height: float,
    heading: str,
    value: str,
    *,
    accent: str,
) -> None:
    x, y = xy
    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.012,rounding_size=0.025",
        transform=ax.transAxes,
        linewidth=1.0,
        edgecolor=accent,
        facecolor=mpl.colors.to_rgba(accent, 0.065),
    )
    ax.add_patch(patch)
    ax.text(x + 0.045 * width, y + 0.69 * height, heading, transform=ax.transAxes,
            ha="left", va="center", fontsize=7.7, color="#62676D")
    ax.text(x + 0.045 * width, y + 0.29 * height, value, transform=ax.transAxes,
            ha="left", va="center", fontsize=10.0, fontweight="bold", color=accent)


def plot_safeguards(ax: mpl.axes.Axes) -> None:
    ax.set_axis_off()
    cram_train = float(confounding.loc[confounding["split"] == "train", "cramers_v_cohort_label"].iloc[0])
    cram_test = float(confounding.loc[confounding["split"] == "test", "cramers_v_cohort_label"].iloc[0])

    add_card(
        ax, (0.00, 0.51), 0.48, 0.43,
        "Cohort × class · Cramér's V",
        f"Dev {cram_train:.3f}",
        accent=CFG.split_colors["train"],
    )
    add_card(
        ax, (0.52, 0.51), 0.48, 0.43,
        "Cohort × class · Cramér's V",
        f"Ext {cram_test:.3f}",
        accent=CFG.split_colors["test"],
    )
    add_card(
        ax, (0.00, 0.02), 1.00, 0.39,
        "External invariance contract",
        f"PASS · max |Δ| = {max_invariance_delta:.1e}",
        accent=CFG.after_color,
    )
    ax.text(
        0.5, -0.13,
        "Descriptive QC only · not used for endpoint, split, model or checkpoint selection",
        transform=ax.transAxes,
        ha="center", va="top",
        fontsize=7.2, color="#6A7077",
    )


## 6. Assemble the supplementary figure


In [ ]:
apply_publication_style(CFG)

fig = plt.figure(figsize=CFG.figure_size)
grid = fig.add_gridspec(
    2,
    4,
    height_ratios=(1.0, 1.03),
    left=0.055,
    right=0.985,
    bottom=0.075,
    top=0.835,
    wspace=0.30,
    hspace=0.50,
)

pca_specs = [
    ("train", "pre_mcombat", "A", "Development · before M-ComBat"),
    ("train", "post_mcombat", "B", "Development · after M-ComBat"),
    ("test", "pre_mcombat", "C", "External · before M-ComBat"),
    ("test", "post_mcombat", "D", "External · projected after M-ComBat"),
]
for col, (split, stage, letter, title) in enumerate(pca_specs):
    ax = fig.add_subplot(grid[0, col])
    plot_pca_panel(ax, split=split, stage=stage, letter=letter, title=title)

ax_metrics = fig.add_subplot(grid[1, :3])
plot_technical_metrics(ax_metrics)

right = grid[1, 3].subgridspec(2, 1, height_ratios=(0.56, 0.44), hspace=0.36)
ax_signal = fig.add_subplot(right[0, 0])
plot_signal_panel(ax_signal)
ax_safeguards = fig.add_subplot(right[1, 0])
plot_safeguards(ax_safeguards)

fig.suptitle(
    "Reference M-ComBat harmonization: cohort mixing and signal preservation",
    x=0.5,
    y=0.985,
    fontsize=CFG.title_size,
    fontweight="bold",
    color=CFG.text_color,
)

cohort_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="none", markersize=6.5,
        markerfacecolor=COHORT_PALETTE[c], markeredgecolor="white",
        markeredgewidth=0.5, label=c,
    )
    for c in cohort_levels
]
fig.legend(
    handles=cohort_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.928),
    ncol=min(7, len(cohort_handles)),
    title="Cohort",
    title_fontsize=CFG.legend_size,
    columnspacing=1.4,
    handletextpad=0.45,
)

fig.text(
    0.5,
    0.027,
    "PCA and all quantitative diagnostics use development-fitted transforms; external samples are projected without refitting.",
    ha="center",
    va="center",
    fontsize=8.2,
    color="#666B70",
)

plt.show()


## 7. Export figure, source values and reproducibility manifest

PDF and SVG retain vector text/axes while embedding the dense PCA point clouds as raster layers. PNG is exported at 600 dpi. TIFF is available through the configuration toggle.


In [ ]:
exported: dict[str, Path] = {}
base = FIGURE_DIR / CFG.output_stem
metadata = {
    "Title": "Reference M-ComBat harmonization: cohort mixing and signal preservation",
    "Subject": "InterGATE Supplementary Figure S1",
    "Creator": "InterGATE notebook 05",
}

if CFG.export_pdf:
    path = base.with_suffix(".pdf")
    fig.savefig(path, bbox_inches="tight", metadata=metadata)
    exported["pdf"] = path

if CFG.export_svg:
    path = base.with_suffix(".svg")
    fig.savefig(path, bbox_inches="tight", metadata={"Title": metadata["Title"]})
    exported["svg"] = path

if CFG.export_png:
    path = base.with_suffix(".png")
    fig.savefig(path, dpi=CFG.dpi, bbox_inches="tight", facecolor="white")
    exported["png"] = path

if CFG.export_tiff:
    path = base.with_suffix(".tiff")
    fig.savefig(
        path,
        dpi=CFG.dpi,
        bbox_inches="tight",
        facecolor="white",
        pil_kwargs={"compression": "tiff_lzw"},
    )
    exported["tiff"] = path

value_frames = [qc.assign(batch_variable=CFG.cohort_col)]
if qc_platform is not None:
    value_frames.append(qc_platform.assign(batch_variable="platform"))
figure_values = pd.concat(value_frames, ignore_index=True)
values_path = FIGURE_DIR / f"{CFG.output_stem}_values.csv"
figure_values.to_csv(values_path, index=False)
exported["values_csv"] = values_path

pca_variance_path = FIGURE_DIR / f"{CFG.output_stem}_pca_variance.csv"
pca_variance.to_csv(pca_variance_path, index=False)
exported["pca_variance_csv"] = pca_variance_path

manifest = {
    "figure": "Supplementary Figure S1",
    "source_notebook": "05_supplementary_figure_mcombat_qc.ipynb",
    "data_prepare": str(DATA_PREPARE),
    "configuration": asdict(CFG),
    "invariance_max_difference": max_invariance_delta,
    "outputs": {key: str(path) for key, path in exported.items()},
    "input_contract": {
        "pre_mcombat": [
            "output_prepared/expr_prepared.csv",
            "output_prepared/metadata_prepared.csv",
        ],
        "post_mcombat": [
            "output_mcombat/expr_train_mcombat.csv",
            "output_mcombat/expr_test_mcombat.csv",
            "output_mcombat/meta_train.csv",
            "output_mcombat/meta_test.csv",
            "output_mcombat/verificacion_invariancia.csv",
        ],
    },
}
manifest_path = FIGURE_DIR / f"{CFG.output_stem}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
exported["manifest_json"] = manifest_path

display(pd.DataFrame([
    {"artifact": key, "path": str(path), "size_mb": path.stat().st_size / 1024**2}
    for key, path in exported.items()
]))


## 8. Manuscript-ready caption and concise Results sentence

The generated sentence reports exact values from the current run; it does not assume that every metric changes in a favourable direction. Retain only statements supported by the exported values table.


In [ ]:
def metric_pair(split: str, metric: str) -> tuple[float, float]:
    return (
        value_at(qc, split, "pre_mcombat", metric),
        value_at(qc, split, "post_mcombat", metric),
    )


caption = (
    "**Supplementary Figure S1 | Reference M-ComBat quality control.** "
    "(A–D) Principal-component projections before and after reference M-ComBat. "
    "For each representation, scaling and PCA were fitted exclusively in the development block; "
    "external samples were projected without refitting. Points denote samples and colours denote cohorts. "
    "(E) Paired technical diagnostics in development and external evaluation: cohort silhouette, "
    "balanced accuracy for predicting cohort from the first 20 PCs, normalized local cohort entropy, "
    "and median and 90th-percentile gene-wise cohort η². Open and filled circles show values before and "
    "after M-ComBat, respectively; desired directions are stated for each metric. "
    "(F) Class-label silhouette, cohort–class association measured by bias-corrected Cramér's V, "
    "and the external invariance check. All diagnostics were descriptive and were not used for endpoint, "
    "split, model, κ, or checkpoint selection."
)

dev_batch = metric_pair("train", "batch_silhouette")
ext_batch = metric_pair("test", "batch_silhouette")
dev_label = metric_pair("train", "label_silhouette")
ext_label = metric_pair("test", "label_silhouette")

results_sentence = (
    "Reference M-ComBat changed the cohort silhouette from "
    f"{dev_batch[0]:.3f} to {dev_batch[1]:.3f} in development and from "
    f"{ext_batch[0]:.3f} to {ext_batch[1]:.3f} in the external projection; "
    "the corresponding class-label silhouettes were "
    f"{dev_label[0]:.3f} versus {dev_label[1]:.3f} and "
    f"{ext_label[0]:.3f} versus {ext_label[1]:.3f}, respectively "
    "(Supplementary Fig. S1)."
)

display(Markdown("### Caption"))
display(Markdown(caption))
display(Markdown("### Optional Results sentence"))
display(Markdown(results_sentence))


## 9. Final automated checks


In [ ]:
required_artifacts = {"values_csv", "pca_variance_csv", "manifest_json"}
if CFG.export_pdf:
    required_artifacts.add("pdf")
if CFG.export_svg:
    required_artifacts.add("svg")
if CFG.export_png:
    required_artifacts.add("png")
if CFG.export_tiff:
    required_artifacts.add("tiff")

missing_exports = required_artifacts.difference(exported)
assert not missing_exports, f"Missing export keys: {sorted(missing_exports)}"

for key in required_artifacts:
    path = exported[key]
    assert path.is_file() and path.stat().st_size > 0, f"Invalid artifact: {path}"

assert set(figure_values["split"]) == {"train", "test"}
assert set(figure_values["stage"]) == {"pre_mcombat", "post_mcombat"}
assert np.isfinite(figure_values.select_dtypes(include=[np.number]).to_numpy()).all(), (
    "The values table contains non-finite numeric values."
)

print("QA PASS")
print(f"External invariance maximum difference: {max_invariance_delta:.3e}")
print(f"Figure directory: {FIGURE_DIR}")

if "png" in exported:
    display(Image(filename=str(exported["png"]), width=1200))
